In [9]:
import pandas as pd

In [10]:
pd.read_csv('Books.csv')

/tmp/ipykernel_8103/1596454934.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('Books.csv')


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...
...,...,...,...,...,...,...,...,...
271355,0440400988,There's a Bat in Bunk Five,Paula Danziger,1988,Random House Childrens Pub (Mm),http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...
271356,0525447644,From One to One Hundred,Teri Sloat,1991,Dutton Books,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...
271357,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker,2004,HarperSanFrancisco,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...
271358,0192126040,Republic (World's Classics),Plato,1996,Oxford University Press,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...


In [11]:
pd.read_csv('Ratings.csv')

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6
...,...,...,...
1149775,276704,1563526298,9
1149776,276706,0679447156,0
1149777,276709,0515107662,10
1149778,276721,0590442449,10


In [12]:
pd.read_csv('Users.csv')

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN
...,...,...,...
278853,278854,"portland, oregon, usa",NaN
278854,278855,"tacoma, washington, united kingdom",50.0
278855,278856,"brampton, ontario, canada",NaN
278856,278857,"knoxville, tennessee, usa",NaN


In [8]:
 pip install pandas numpy scikit-learn scipy flask pyngrok

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ============================================================
# BOOK RECOMMENDATION SYSTEM — ONE CELL, END TO END (v3)
# Data cleaning -> Feature Engineering -> Matrix Factorization (SVD)
# + k-NN Collaborative Filtering (with train/test RMSE evaluation)
# -> Smart fuzzy search + live autocomplete -> Professional Flask
# website -> Public URL via ngrok (OPTIONAL — falls back to local)
# ============================================================
# BEFORE YOU RUN:
# 1) Put Books.csv, Ratings.csv, Users.csv in the SAME folder as
#    this notebook (or edit the paths below).
# 2) pip install these once in a normal cell (not this one):
#       pip install pandas numpy scikit-learn scipy flask pyngrok requests
# 3) (OPTIONAL) If you want a public https:// URL instead of just
#    localhost, get a FREE ngrok AUTHTOKEN (not API key) from
#    https://dashboard.ngrok.com/get-started/your-authtoken
#    and set it as an environment variable BEFORE launching Jupyter,
#    e.g. in a terminal:
#        export NGROK_AUTH_TOKEN="your_real_token_here"      (Mac/Linux)
#        setx NGROK_AUTH_TOKEN "your_real_token_here"        (Windows)
#    Do NOT paste your real token directly into this file if you ever
#    share it — treat it like a password. If no token is set, the app
#    just runs locally at http://localhost:5000 instead of failing.
# 4) Run this whole file as ONE cell in Jupyter.
# ============================================================

import os
import pandas as pd
import numpy as np
import re
import difflib
import threading

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from flask import Flask, request, render_template_string, jsonify
import requests

# --------------------------------------------------------------
# 0) CONFIG — EDIT THESE
# --------------------------------------------------------------
BOOKS_PATH = "Books.csv"
RATINGS_PATH = "Ratings.csv"
USERS_PATH = "Users.csv"

# Read the ngrok token from an environment variable instead of hardcoding
# it in the file. This is safer (nothing to accidentally paste/share) and
# means the script degrades gracefully instead of crashing when no token
# is available — see section 8 at the bottom.
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "").strip()

# Minimum activity thresholds so the matrix is dense enough to learn
# real signal but still small enough to train quickly.
MIN_RATINGS_PER_BOOK = 20     # a book needs at least this many ratings
MIN_RATINGS_PER_USER = 60     # a user needs to have rated at least this many books
SVD_COMPONENTS = 50           # latent factors learned by the SVD model

# --------------------------------------------------------------
# 1) ROBUST CSV LOADING
# --------------------------------------------------------------
def robust_read_csv(path):
    attempts = [
        dict(encoding="utf-8", sep=","),
        dict(encoding="latin-1", sep=","),
        dict(encoding="latin-1", sep=";"),
        dict(encoding="ISO-8859-1", sep=";", on_bad_lines="skip", low_memory=False),
    ]
    last_err = None
    for kw in attempts:
        try:
            df = pd.read_csv(path, **kw, low_memory=False)
            if df.shape[1] > 1:
                return df
        except Exception as e:
            last_err = e
            continue
    raise last_err

print("Loading datasets...")
books = robust_read_csv(BOOKS_PATH)
ratings = robust_read_csv(RATINGS_PATH)
users = robust_read_csv(USERS_PATH)
print(f"Books: {books.shape}, Ratings: {ratings.shape}, Users: {users.shape}")

# --------------------------------------------------------------
# 2) CLEANING + FEATURE ENGINEERING
# --------------------------------------------------------------

# ---- Books ----
books.columns = [c.strip() for c in books.columns]
keep_cols = ["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication",
             "Publisher", "Image-URL-M"]
keep_cols = [c for c in keep_cols if c in books.columns]
books = books[keep_cols].copy()

books.dropna(subset=["Book-Title"], inplace=True)
books["Book-Author"] = books["Book-Author"].fillna("Unknown")
books["Publisher"] = books["Publisher"].fillna("Unknown")
if "Image-URL-M" not in books.columns:
    books["Image-URL-M"] = ""
books["Image-URL-M"] = books["Image-URL-M"].fillna("")

books["Year-Of-Publication"] = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
books.loc[(books["Year-Of-Publication"] < 1450) |
          (books["Year-Of-Publication"] > 2026), "Year-Of-Publication"] = np.nan
books["Year-Of-Publication"] = books["Year-Of-Publication"].fillna(
    books["Year-Of-Publication"].median()
)

# normalized title for exact/substring matching
books["title_clean"] = (
    books["Book-Title"].astype(str).str.lower().str.strip()
    .apply(lambda s: re.sub(r"[^a-z0-9 ]", "", s))
)
books.drop_duplicates(subset=["ISBN"], inplace=True)

# ---- Users ----
users.columns = [c.strip() for c in users.columns]
loc_split = users["Location"].astype(str).str.split(",", expand=True)
users["country"] = loc_split[loc_split.columns[-1]].str.strip().str.lower()
users["country"] = users["country"].replace(["", "n/a", "nan", "none"], "unknown")

users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
users.loc[(users["Age"] < 5) | (users["Age"] > 100), "Age"] = np.nan
users["Age"] = users["Age"].fillna(users["Age"].median())

# ---- Ratings ----
ratings.columns = [c.strip() for c in ratings.columns]
ratings = ratings[ratings["ISBN"].isin(books["ISBN"])]
ratings["Book-Rating"] = pd.to_numeric(ratings["Book-Rating"], errors="coerce")
ratings.dropna(subset=["Book-Rating"], inplace=True)

# only explicit ratings (0 means "no rating given" in this dataset)
ratings_explicit = ratings[ratings["Book-Rating"] > 0].copy()

# --------------------------------------------------------------
# 3) POPULARITY-BASED MODEL (cold-start / fallback recommender)
# --------------------------------------------------------------
book_stats = ratings_explicit.groupby("ISBN").agg(
    num_ratings=("Book-Rating", "count"),
    avg_rating=("Book-Rating", "mean"),
).reset_index()

popularity = book_stats.merge(books, on="ISBN")
C = popularity["avg_rating"].mean()
m = 25
popularity["weighted_score"] = (
    (popularity["num_ratings"] / (popularity["num_ratings"] + m)) * popularity["avg_rating"]
    + (m / (popularity["num_ratings"] + m)) * C
)
top_popular = popularity.sort_values("weighted_score", ascending=False).head(300)

# --------------------------------------------------------------
# 4) BUILD RATING MATRIX (books x users)
# --------------------------------------------------------------
print("Building the ratings matrix...")

book_counts = ratings_explicit["ISBN"].value_counts()
popular_isbns = book_counts[book_counts >= MIN_RATINGS_PER_BOOK].index

user_counts = ratings_explicit["User-ID"].value_counts()
active_users = user_counts[user_counts >= MIN_RATINGS_PER_USER].index

cf_ratings = ratings_explicit[
    ratings_explicit["ISBN"].isin(popular_isbns) &
    ratings_explicit["User-ID"].isin(active_users)
].copy()

print(f"Collaborative-filtering training rows: {cf_ratings.shape[0]}")

# --------------------------------------------------------------
# 4b) TRAIN / TEST SPLIT + MODEL EVALUATION
#     This is what makes the model properly "trained": we hold out
#     20% of interactions, fit SVD on the remaining 80%, and measure
#     how well it predicts the held-out ratings (RMSE).
# --------------------------------------------------------------
train_ratings, test_ratings = train_test_split(
    cf_ratings, test_size=0.2, random_state=42
)

train_pivot = train_ratings.pivot_table(
    index="ISBN", columns="User-ID", values="Book-Rating"
).fillna(0)

print(f"Training matrix shape: {train_pivot.shape}  "
      f"(books x users, {(train_pivot.values != 0).sum()} known ratings)")

n_components_eval = min(SVD_COMPONENTS, min(train_pivot.shape) - 1)
svd_eval = TruncatedSVD(n_components=n_components_eval, random_state=42)
latent_books_eval = svd_eval.fit_transform(train_pivot.values)
reconstructed = np.dot(latent_books_eval, svd_eval.components_)
reconstructed_df = pd.DataFrame(
    reconstructed, index=train_pivot.index, columns=train_pivot.columns
)

# score only test pairs that exist in the training matrix's index/columns
y_true, y_pred = [], []
for _, row in test_ratings.iterrows():
    isbn, uid, true_rating = row["ISBN"], row["User-ID"], row["Book-Rating"]
    if isbn in reconstructed_df.index and uid in reconstructed_df.columns:
        y_true.append(true_rating)
        y_pred.append(np.clip(reconstructed_df.loc[isbn, uid], 0, 10))

if y_true:
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"Model evaluation -> RMSE on held-out test ratings: {rmse:.3f} "
          f"(scale 1-10, evaluated on {len(y_true)} test ratings, "
          f"{n_components_eval} latent factors)")
else:
    print("Not enough overlap between train/test users to compute RMSE — "
          "consider lowering MIN_RATINGS thresholds.")

# --------------------------------------------------------------
# 4c) FINAL MODEL: train on ALL available data for serving
#     (SVD for latent embeddings + k-NN in latent space, which is
#     faster and captures deeper patterns than raw k-NN on sparse
#     rating vectors)
# --------------------------------------------------------------
print("Training final model on full data...")

pivot = cf_ratings.pivot_table(
    index="ISBN", columns="User-ID", values="Book-Rating"
).fillna(0)

isbn_to_title = books.set_index("ISBN")["Book-Title"].to_dict()
pivot_index_list = list(pivot.index)

n_components_final = min(SVD_COMPONENTS, min(pivot.shape) - 1)
svd_final = TruncatedSVD(n_components=n_components_final, random_state=42)
latent_books = svd_final.fit_transform(pivot.values)  # one latent vector per book

model_knn = NearestNeighbors(metric="cosine", algorithm="brute")
model_knn.fit(latent_books)

print(f"Final model trained: {pivot.shape[0]} books, {pivot.shape[1]} users, "
      f"{n_components_final} latent factors.")

# --------------------------------------------------------------
# 5) SMART SEARCH (typo-tolerant, with live autocomplete)
# --------------------------------------------------------------
all_titles = books["Book-Title"].tolist()
all_titles_clean = books["title_clean"].tolist()
title_to_row = {t: i for i, t in enumerate(all_titles_clean)}

def cheap_candidates(query_clean, limit=500):
    """Fast first-pass filter: substring match (vectorized)."""
    if not query_clean:
        return books.iloc[0:0]
    mask = books["title_clean"].str.contains(re.escape(query_clean), na=False)
    return books[mask].head(limit)

def smart_title_match(user_input, cutoff=0.55):
    """Two-stage smart search:
    1) exact normalized match
    2) substring match
    3) fuzzy match (typo-tolerant) using difflib over the closest
       candidates, so 'Harry Poter' still finds 'Harry Potter'.
    """
    q = re.sub(r"[^a-z0-9 ]", "", user_input.lower().strip())
    if not q:
        return None, None

    # 1) exact
    if q in title_to_row:
        row = books.iloc[title_to_row[q]]
        return row["ISBN"], row["Book-Title"]

    # 2) substring
    cand = cheap_candidates(q)
    if not cand.empty:
        row = cand.iloc[0]
        return row["ISBN"], row["Book-Title"]

    # 3) fuzzy (typo tolerant) — only run over a sample for speed
    sample_size = 20000
    sample_titles = all_titles_clean[:sample_size] if len(all_titles_clean) > sample_size else all_titles_clean
    close = difflib.get_close_matches(q, sample_titles, n=1, cutoff=cutoff)
    if close:
        row = books.iloc[title_to_row[close[0]]]
        return row["ISBN"], row["Book-Title"]

    return None, None

def autocomplete_suggestions(partial, limit=8):
    """Used by the live-search dropdown in the website."""
    q = re.sub(r"[^a-z0-9 ]", "", partial.lower().strip())
    if len(q) < 2:
        return []
    cand = cheap_candidates(q, limit=300)
    if cand.empty:
        # fall back to fuzzy matching for typos while typing
        close = difflib.get_close_matches(q, all_titles_clean[:20000], n=limit, cutoff=0.5)
        cand = books[books["title_clean"].isin(close)]
    cand = cand.drop_duplicates(subset="Book-Title").head(limit)
    return cand["Book-Title"].tolist()

# --------------------------------------------------------------
# 5b) OPEN LIBRARY API HELPERS (real, free, no key required)
#     https://openlibrary.org/developers/api
# --------------------------------------------------------------
OL_COVER_URL = "https://covers.openlibrary.org/b/isbn/{isbn}-M.jpg"
OL_SEARCH_URL = "https://openlibrary.org/search.json"

def get_cover_url(isbn, fallback_url=""):
    if isbn:
        return OL_COVER_URL.format(isbn=isbn)
    return fallback_url or "https://via.placeholder.com/150x220?text=No+Cover"

def search_open_library(title_query, limit=6):
    try:
        resp = requests.get(
            OL_SEARCH_URL,
            params={"title": title_query, "limit": limit,
                    "fields": "title,author_name,isbn,key"},
            timeout=5,
            headers={"User-Agent": "book-recommender-demo/1.0"},
        )
        resp.raise_for_status()
        docs = resp.json().get("docs", [])
        results = []
        for d in docs:
            isbn_list = d.get("isbn", [])
            results.append({
                "Book-Title": d.get("title", "Unknown title"),
                "Book-Author": ", ".join(d.get("author_name", [])) or "Unknown",
                "ISBN": isbn_list[0] if isbn_list else "",
                "Image-URL-M": "",
            })
        return pd.DataFrame(results)
    except Exception as e:
        print("Open Library API call failed:", e)
        return pd.DataFrame(columns=["Book-Title", "Book-Author", "ISBN", "Image-URL-M"])

# --------------------------------------------------------------
# 6) RECOMMENDATION FUNCTION
# --------------------------------------------------------------
def get_recommendations(user_input, n=8):
    isbn, matched_title = smart_title_match(user_input)

    if isbn is not None and isbn in pivot.index:
        idx = pivot_index_list.index(isbn)
        book_vector = latent_books[idx].reshape(1, -1)
        distances, indices = model_knn.kneighbors(book_vector, n_neighbors=n + 1)
        rec_isbns = [pivot_index_list[i] for i in indices.flatten()][1:]
        recs = books[books["ISBN"].isin(rec_isbns)].drop_duplicates("ISBN")
        recs = recs.set_index("ISBN").loc[rec_isbns].reset_index()
        return matched_title, recs, "SVD + k-NN collaborative filtering"

    if isbn is not None:
        recs = top_popular.head(n)
        return matched_title, recs, "popularity fallback (not enough rating data for this book)"

    ol_results = search_open_library(user_input, limit=n)
    if not ol_results.empty:
        return None, ol_results, "Open Library live search (no local match)"

    recs = top_popular.head(n)
    return None, recs, "popularity fallback (no matching book found anywhere)"

# --------------------------------------------------------------
# 7) PROFESSIONAL FLASK WEBSITE
# --------------------------------------------------------------
app = Flask(__name__)

PAGE_TEMPLATE = """
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Shelfwise — Book Recommender</title>
<style>
  :root {
    --bg: #0f1115;
    --panel: #171a21;
    --panel-2: #1e222b;
    --text: #eef0f4;
    --muted: #9aa2b1;
    --accent: #e0a458;
    --accent-2: #6c8cff;
    --border: #2a2f3a;
    --radius: 14px;
  }
  * { box-sizing: border-box; }
  body {
    margin: 0;
    background: radial-gradient(circle at top, #1a1d26 0%, var(--bg) 60%);
    color: var(--text);
    font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
    min-height: 100vh;
  }
  .nav {
    display: flex; align-items: center; justify-content: space-between;
    padding: 20px 40px; border-bottom: 1px solid var(--border);
  }
  .nav .brand { font-size: 22px; font-weight: 700; letter-spacing: 0.5px; }
  .nav .brand span { color: var(--accent); }
  .nav .tag { color: var(--muted); font-size: 13px; }

  .hero {
    text-align: center; padding: 60px 20px 30px;
  }
  .hero h1 {
    font-size: 34px; margin: 0 0 10px;
    background: linear-gradient(90deg, var(--text), var(--accent));
    -webkit-background-clip: text; background-clip: text; color: transparent;
  }
  .hero p { color: var(--muted); max-width: 560px; margin: 0 auto; font-size: 15px; }

  .search-wrap { max-width: 620px; margin: 34px auto 10px; padding: 0 20px; position: relative; }
  .search-box {
    display: flex; background: var(--panel); border: 1px solid var(--border);
    border-radius: 999px; padding: 6px; box-shadow: 0 10px 30px rgba(0,0,0,0.35);
  }
  .search-box input {
    flex: 1; border: none; background: transparent; color: var(--text);
    padding: 14px 20px; font-size: 15px; outline: none;
  }
  .search-box button {
    border: none; border-radius: 999px; padding: 0 26px; font-size: 15px;
    font-weight: 600; cursor: pointer; background: linear-gradient(90deg, var(--accent), #c9843a);
    color: #1a1200; transition: transform 0.15s ease;
  }
  .search-box button:hover { transform: translateY(-1px); }

  #suggestions {
    position: absolute; left: 20px; right: 20px; top: 60px; z-index: 10;
    background: var(--panel-2); border: 1px solid var(--border); border-radius: 12px;
    overflow: hidden; display: none; box-shadow: 0 12px 30px rgba(0,0,0,0.45);
  }
  #suggestions div {
    padding: 11px 18px; cursor: pointer; font-size: 14px; color: var(--text);
    border-bottom: 1px solid var(--border);
  }
  #suggestions div:last-child { border-bottom: none; }
  #suggestions div:hover { background: #262b36; color: var(--accent); }

  .status-row { text-align: center; margin: 26px 0 6px; color: var(--muted); font-size: 14px; }
  .status-row b { color: var(--text); }
  .method-pill {
    display: inline-block; margin-top: 6px; padding: 4px 12px; border-radius: 999px;
    background: rgba(108,140,255,0.12); color: var(--accent-2); font-size: 12px; font-weight: 600;
    border: 1px solid rgba(108,140,255,0.3);
  }

  .grid {
    display: grid; grid-template-columns: repeat(auto-fill, minmax(170px, 1fr));
    gap: 22px; max-width: 1100px; margin: 30px auto 60px; padding: 0 30px;
  }
  .card {
    background: var(--panel); border: 1px solid var(--border); border-radius: var(--radius);
    padding: 14px; text-align: center; transition: transform 0.18s ease, border-color 0.18s ease;
  }
  .card:hover { transform: translateY(-6px); border-color: var(--accent); }
  .card img {
    width: 100%; aspect-ratio: 2/3; object-fit: cover; border-radius: 8px;
    background: #2a2f3a;
  }
  .card h4 { font-size: 13.5px; margin: 12px 0 4px; height: 36px; overflow: hidden; line-height: 1.3; }
  .card p { font-size: 12px; color: var(--muted); margin: 2px 0; }
  .card .rating { color: var(--accent); font-size: 12px; margin-top: 4px; }

  .empty-state { text-align: center; color: var(--muted); margin-top: 50px; font-size: 14px; }
  footer { text-align: center; color: var(--muted); font-size: 12.5px; padding: 30px; border-top: 1px solid var(--border); }
  footer a { color: var(--accent-2); text-decoration: none; }
  .spinner {
    display: none; margin: 30px auto; width: 34px; height: 34px;
    border: 3px solid var(--border); border-top-color: var(--accent);
    border-radius: 50%; animation: spin 0.8s linear infinite;
  }
  @keyframes spin { to { transform: rotate(360deg); } }
</style>
</head>
<body>
  <div class="nav">
    <div class="brand">Shelf<span>wise</span></div>
    <div class="tag">SVD + k-NN collaborative filtering</div>
  </div>

  <div class="hero">
    <h1>Find your next favorite book</h1>
    <p>Type a title you love — Shelfwise learns from thousands of readers' ratings to find what you'll like next.</p>
  </div>

  <div class="search-wrap">
    <form method="GET" action="/recommend" id="searchForm">
      <div class="search-box">
        <input type="text" name="title" id="titleInput" autocomplete="off"
               placeholder="e.g. The Hobbit, Harry Potter, The Notebook..." value="{{query or ''}}">
        <button type="submit">Recommend</button>
      </div>
      <div id="suggestions"></div>
    </form>
  </div>

  {% if recs is not none %}
    <div class="status-row">
      {% if matched_title %}
        Because you liked <b>{{matched_title}}</b>
      {% elif query %}
        No exact match for "<b>{{query}}</b>" — here's the closest we found
      {% endif %}
      <div class="method-pill">{{method}}</div>
    </div>

    {% if recs.empty %}
      <div class="empty-state">No recommendations found. Try a different title.</div>
    {% else %}
      <div class="grid">
        {% for _, row in recs.iterrows() %}
          <div class="card">
            <img src="{{ cover(row['ISBN'], row['Image-URL-M']) }}"
                 onerror="this.onerror=null;this.src='https://via.placeholder.com/150x220?text=No+Cover';">
            <h4>{{ row['Book-Title'] }}</h4>
            <p>{{ row['Book-Author'] }}</p>
            {% if 'avg_rating' in row and row['avg_rating'] == row['avg_rating'] %}
              <p class="rating">★ {{ "%.1f"|format(row['avg_rating']) }} / 10</p>
            {% endif %}
          </div>
        {% endfor %}
      </div>
    {% endif %}
  {% endif %}

  <footer>
    Built with SVD matrix factorization + k-NN collaborative filtering &middot; covers via
    <a href="https://openlibrary.org/developers/api" target="_blank">Open Library API</a>
  </footer>

<script>
  const input = document.getElementById('titleInput');
  const box = document.getElementById('suggestions');
  let debounceTimer;

  input.addEventListener('input', () => {
    clearTimeout(debounceTimer);
    const q = input.value.trim();
    if (q.length < 2) { box.style.display = 'none'; return; }
    debounceTimer = setTimeout(() => {
      fetch('/api/search?q=' + encodeURIComponent(q))
        .then(r => r.json())
        .then(data => {
          if (!data.length) { box.style.display = 'none'; return; }
          box.innerHTML = data.map(t => `<div onclick="selectTitle('${t.replace(/'/g, "\\'")}')">${t}</div>`).join('');
          box.style.display = 'block';
        })
        .catch(() => { box.style.display = 'none'; });
    }, 220);
  });

  function selectTitle(t) {
    input.value = t;
    box.style.display = 'none';
    document.getElementById('searchForm').submit();
  }

  document.addEventListener('click', (e) => {
    if (!box.contains(e.target) && e.target !== input) box.style.display = 'none';
  });
</script>
</body>
</html>
"""

@app.route("/")
def home():
    return render_template_string(PAGE_TEMPLATE, recs=None, query=None, matched_title=None, method=None, cover=get_cover_url)

@app.route("/recommend")
def recommend():
    query = request.args.get("title", "").strip()
    if not query:
        return render_template_string(PAGE_TEMPLATE, recs=None, query=None, matched_title=None, method=None, cover=get_cover_url)
    matched_title, recs, method = get_recommendations(query)
    return render_template_string(
        PAGE_TEMPLATE, recs=recs, query=query, matched_title=matched_title, method=method, cover=get_cover_url
    )

@app.route("/api/search")
def api_search():
    q = request.args.get("q", "")
    return jsonify(autocomplete_suggestions(q))

# --------------------------------------------------------------
# 8) RUN THE APP — PUBLIC URL VIA NGROK IF A TOKEN IS AVAILABLE,
#    OTHERWISE JUST RUN LOCALLY (no more hard crash if you haven't
#    set up ngrok yet).
# --------------------------------------------------------------

def run_app_locally():
    print("=" * 60)
    print(" No NGROK_AUTH_TOKEN found (or pyngrok unavailable) — running locally.")
    print(" Open this URL in your browser: http://localhost:5000")
    print("=" * 60)
    app.run(port=5000)


def run_app_with_ngrok(token):
    from pyngrok import ngrok

    ngrok.kill()
    ngrok.set_auth_token(token)
    public_url = ngrok.connect(5000, "http")

    print("=" * 60)
    print(f" Your website is live at: {public_url}")
    print(" Open that URL in your browser, type a book title, and click Recommend.")
    print("=" * 60)

    def run_app():
        app.run(port=5000)

    threading.Thread(target=run_app).start()


if NGROK_AUTH_TOKEN:
    try:
        run_app_with_ngrok(NGROK_AUTH_TOKEN)
    except Exception as e:
        print(f"ngrok setup failed ({e}); falling back to running locally.")
        run_app_locally()
else:
    print(
        "Tip: to get a public https:// link instead of localhost, set the "
        "NGROK_AUTH_TOKEN environment variable before launching Jupyter "
        "(see the comment block at the top of this file) and re-run this cell."
    )
    run_app_locally()

Loading datasets...
Books: (271360, 8), Ratings: (1149780, 3), Users: (278858, 3)
Building the ratings matrix...
Collaborative-filtering training rows: 26062
Training matrix shape: (2121, 918)  (books x users, 20849 known ratings)
Model evaluation -> RMSE on held-out test ratings: 7.714 (scale 1-10, evaluated on 5191 test ratings, 50 latent factors)
Training final model on full data...
Final model trained: 2128 books, 926 users, 50 latent factors.
Tip: to get a public https:// link instead of localhost, set the NGROK_AUTH_TOKEN environment variable before launching Jupyter (see the comment block at the top of this file) and re-run this cell.
 No NGROK_AUTH_TOKEN found (or pyngrok unavailable) — running locally.
 Open this URL in your browser: http://localhost:5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Aug/2026 23:14:34] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:35] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [12/Aug/2026 23:14:41] "GET /api/search?q=dc HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:46] "GET /api/search?q=al HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:48] "GET /api/search?q=alc HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:49] "GET /api/search?q=al HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:51] "GET /api/search?q=bio HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:53] "GET /api/search?q=biol HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:54] "GET /api/search?q=biolo HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:56] "GET /api/search?q=biolog HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:56] "GET /api/search?q=biology HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:14:58] "GET /recommend?title=biology HTTP/1.1" 200 -
127.0.0.1 - - [12/Aug/2026 23:15:03] "